# 02 — Feature Engineering
**Dynamic Pricing Engine** | Mohit | github.com/dswithmohit/dynamic-pricing-engine

---
### Features built in this notebook
| Feature | Description |
|---|---|
| `competitor_price_ratio` | unit_price / competitor_price — competitive positioning |
| `demand_lag_7` | units_sold lagged 7 days — short-term momentum |
| `demand_lag_14` | units_sold lagged 14 days — medium-term trend |
| `rolling_demand_28` | 28-day rolling mean — smoothed demand signal |
| `seasonality_index` | sinusoidal month signal ∈ [0, 1] |
| `inventory_level` | simulated stock on hand — supply constraint |
| `price_to_avg_category` | item price / daily category average |
| `is_weekend`, `quarter` | calendar features |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

from src.data_loader import load_raw, build_synthetic_dataset
from src.features import build_features, feature_summary
from src.config import PROC_DIR, PROCESSED_CSV

## 1. Load Synthetic Dataset

In [ ]:
try:
    df_raw = load_raw()
except FileNotFoundError:
    from src.data_loader import _make_seed_df
    df_raw = _make_seed_df()

df_synth = build_synthetic_dataset(df_raw, target_rows=500_000)
print(df_synth.shape)

## 2. Run Feature Pipeline

In [ ]:
df_feat = build_features(df_synth)
feature_summary(df_feat)
df_feat.head(3)

## 3. Competitor Price Ratio Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_feat['competitor_price_ratio'].clip(0.5, 1.5), bins=60,
        color='#2E75B6', edgecolor='white')
ax.axvline(1.0, color='#C00000', linestyle='--', linewidth=1.5, label='Parity (ratio=1)')
ax.set_title('Competitor Price Ratio Distribution')
ax.set_xlabel('unit_price / competitor_price')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/competitor_price_ratio.png', bbox_inches='tight')
plt.show()
print(df_feat['competitor_price_ratio'].describe().round(3))

## 4. Seasonality Index

In [ ]:
months = range(1, 13)
import numpy as np
si = [round(0.5 * (1 + np.sin(2 * np.pi * (m - 3) / 12)), 4) for m in months]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(list(months), si, marker='o', color='#70AD47', linewidth=2)
ax.set_xticks(list(months))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_title('Seasonality Index by Month (sinusoidal, peak = Oct)')
ax.set_ylabel('Seasonality Index')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/seasonality_index.png', bbox_inches='tight')
plt.show()

## 5. Demand Lag Correlation

In [ ]:
lag_cols = ['units_sold', 'demand_lag_7', 'demand_lag_14', 'rolling_demand_28']
lag_cols_present = [c for c in lag_cols if c in df_feat.columns]
corr = df_feat[lag_cols_present].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Demand Lag Feature Correlations')
plt.tight_layout()
plt.show()

## 6. Inventory Level Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_feat['inventory_level'].clip(upper=df_feat['inventory_level'].quantile(0.99)),
        bins=50, color='#ED7D31', edgecolor='white')
ax.set_title('Inventory Level Distribution')
ax.set_xlabel('Inventory Level (units)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 7. Save Feature Matrix

In [ ]:
df_feat.to_csv(PROCESSED_CSV, index=False)
print(f'Feature matrix saved → {PROCESSED_CSV}')
print(f'Shape: {df_feat.shape}')

## Key Takeaways
- `competitor_price_ratio` is **centred near 1.0** — most products are priced competitively
- `demand_lag_7` and `demand_lag_14` have high autocorrelation — strong temporal persistence
- `seasonality_index` captures Oct–Dec peak demand period
- `inventory_level` adds supply-side signal — low stock correlates with higher prices

➡ Proceed to `03_elasticity_modeling.ipynb`